In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy


from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
# data = sc.read_h5ad(data_annotated_path + "joint_annotated.h5ad")
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden.h5ad")

data.shape

In [ ]:
# CHECK POPULATION LOSS DUMBASS
# RODS CONES 
# METHODS 
# GSEA ON THE DIFF 

data.obs['time']=data.obs['time'].astype(int)

# Building A Timeline

In [ ]:
plt.figure()
plt.title("Log total counts vs time")
plt.boxplot(
    [data.obs['log_total_counts'][data.obs['time'] == t] for t in [12,24,48,72]],
    showfliers=False,
)
plt.xticks(
    [1,2,3,4],
    ["12h","24h","48h","72h"]
)
plt.ylabel("Log total counts")
plt.show()


# PCA IQR Timeline, Big Sheet

We can briefly look at the timecourse of each individual PC across the entire dataset with no filtering to sanity check our data and see the overall level of stability. We also want to observe whether or not there are obvious outliers

In [ ]:
def extract_iqrs(data,low_percentile=25,high_percentile=75):
    return {
        'low':np.percentile(data,low_percentile),
        'center':np.median(data),
        'high':np.percentile(data,high_percentile),
    }

def extract_sems(data,sem_multiple=2):
    sem = scipy.stats.sem(data)
    mean = np.mean(data)
    return {
        'low':mean - (sem*sem_multiple),
        'center':mean,
        'high':mean + (sem*sem_multiple),
    }

def filled_centerline_to_ax(
    ax,#times,
    shadowed_segments,
    label=None,fill_label=None,ax_label=None,
    **kwargs
):

    centerline_to_ax(ax,shadowed_segments,label=label,ax_label=ax_label)

    times = sorted(shadowed_segments.keys())
    ax.fill_between(
        times,
        [shadowed_segments[t]['low'] for t in times],
        [shadowed_segments[t]['high'] for t in times],
        label=fill_label,alpha=.2
    )

    if fill_label:
        ax.legend()
    return ax

def centerline_to_ax(
    ax,
    shadowed_segments,
    label=None,ax_label=None,suppress_legend=False,
    **kwargs
):
    times = sorted(shadowed_segments.keys())

    ax.plot(
        times,
        [shadowed_segments[t]['center'] for t in times],
        label=label,
        **kwargs
    )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel(ax_label)
    ax.set_title(ax_label)
    if label and not suppress_legend:
        ax.legend()
    return ax

def get_compound_masks(
    data
):
    times = sorted(set(data.obs['time']))
    time_masks = [
        data.obs['time'] == t
        for t in times
    ]

    control_mask =  data.obs['exp_condition'] == "Cntr"
    injury_mask = data.obs['exp_condition'] == "Mtz"

    combined_masks = {
        'control':{
            time:control_mask & time_mask for time,time_mask in zip(times,time_masks)
        },
        'injury':{
            time: injury_mask & time_mask for time,time_mask in zip(times,time_masks)
        }
    }

    return combined_masks

def map_over_leaves(compound,fn):
   return {
    outer_key: {inner_key: fn(v) for inner_key, v in inner.items()} 
    for outer_key, inner in compound.items()
} 

def get_compound_scores(
    compound_masks,target
):
    return map_over_leaves(compound_masks,lambda mask: target[mask])

def get_compound_iqrs(
    compound_scores
):
    return map_over_leaves(compound_scores, extract_iqrs)

def get_compound_means(
    compound_scores
):
    return map_over_leaves(compound_scores, extract_sems)


def plot_timecourse_from_compound(
    compound_iqrs,ax,
    label=None,fill_label=None,ax_label=None,
    plot_shadow=True,suppress_legend=False,alpha=1,
):
    for condition in compound_iqrs:

        if plot_shadow:
            filled_centerline_to_ax(
                ax,compound_iqrs[condition],
                label=f"{label} : {condition}",
                fill_label=fill_label,
                ax_label=ax_label,suppress_legend=suppress_legend,alpha=alpha
            )
        else:
            centerline_to_ax(
                ax,compound_iqrs[condition],
                label=f"{label} : {condition}",
                ax_label=ax_label,suppress_legend=suppress_legend,alpha=alpha
            )

def plot_timecourse(
    target,compound_masks,ax,
    label=None,fill_label=None,ax_label=None,central_tendency="median",
    plot_shadow=True,suppress_legend=False,alpha=1
):
    compound_scores = get_compound_scores(compound_masks,target)
    compound_shadow = None
    shadow_label = None
    match central_tendency:
        case "median": 
            compound_shadow = get_compound_iqrs(compound_scores)
            shadow_label = "IQR"
        case "mean": 
            compound_shadow = get_compound_means(compound_scores)
            shadow_label = "SEM"
        case _: raise ValueError(f"Invalid central tendency: {central_tendency}")
    plot_timecourse_from_compound(
        compound_shadow,ax,
        fill_label=shadow_label,label=label,
        ax_label=ax_label,plot_shadow=plot_shadow,suppress_legend=suppress_legend,alpha=alpha
    )

In [ ]:
compound_masks = get_compound_masks(data)

fig,axes = plt.subplots(5,10,figsize=(30,15))
for i in range(50):
    ax = axes[i//10,i%10]

    target = data.obsm['X_pca'][:,i]
    plot_timecourse(
        target,compound_masks,ax,
        label=f"PC{i}",
        ax_label=f'PC{i}',central_tendency="median"
    )
fig.tight_layout()
fig.show()


# Cell Death Scoring

The particulars of producing these scores are in the sister file "annotations", but this is the set of gene sets used by the TBI guys. They can be updated with alternatives on demand. (Swap in the gene sets and map the homologs to zebrafish genome)

We want to plot the timecourse of the score, as well as the scores over the UMAP to see if there are any obvious patterns.

Brief note: representing an uncertainty score here is a little fraught, I'm not a huge fan of either the IQR or the SEM, the former is too vague and the latter is too conservative (as written it is treating each cell as a unique observation)

I want to check if there are biological replicate annotations, if so we can re-calculate the SEM on a per-sample basis, which will better represent the actual variation certainty. Please stand by for this.


In [ ]:
parthanatos_mask = np.array(data.var['parthanatos'])
apoptosis_mask = np.array(data.var['apoptosis'])
necroptosis_mask = np.array(data.var['necroptosis'])

parthanatos_signature = data.var_names[parthanatos_mask]
apoptosis_signature = data.var_names[apoptosis_mask]
necroptosis_signature = data.var_names[necroptosis_mask]

print(f"Parthanatos: {parthanatos_signature}")
print(f"Necroptosis: {necroptosis_signature}")
print(f"Apoptosis: {apoptosis_signature}")

sc.tl.score_genes(data,parthanatos_signature,score_name="parthanatos")
sc.tl.score_genes(data,necroptosis_signature,score_name="necroptosis")
sc.tl.score_genes(data,apoptosis_signature,score_name="apoptosis")


### Sham Signature 

In [ ]:
# Let's also construct a sham gene signature to get a sense for "baseline" signal

# First we want to establish the overall number of genes in ext signatures:

print((
    f'Parthanatos: {np.sum(data.var['parthanatos'])}\n'
    f'Necroptosis: {np.sum(data.var['necroptosis'])}\n'
    f'Apoptosis: {np.sum(data.var['apoptosis'])}\n'
))

In [ ]:
# n_features = len(data.var_names)
# n_shams = 5
# small_sham_signature_masks = [np.random.randint(0,n_features,size=10) for _ in range(n_shams)]
# medium_sham_signature_masks = [np.random.randint(0,n_features,size=17) for _ in range(n_shams)]
# large_sham_signature_masks = [np.random.randint(0,n_features,size=24) for _ in range(n_shams)]

# for i,sham in enumerate(small_sham_signature_masks + medium_sham_signature_masks + large_sham_signature_masks):
#     sc.tl.score_genes(data,data.var_names[sham],score_name=f'sham_{i}')
    



In [ ]:
compound_masks = get_compound_masks(data)
n_features = len(data.var_names)

lotta_sham = [np.random.randint(0,n_features,size=17) for _ in range(1000)]
sham_timecourses = []

# plt.figure(figsize=(10,10))
for i,sham in enumerate(lotta_sham):
    print(f"sham {i}")
    sc.tl.score_genes(data,data.var_names[sham],score_name=f'sham_tmp')
    scores = get_compound_scores(compound_masks,data.obs['sham_tmp'])
    iqrs = get_compound_iqrs(scores)
    sham_timecourses.append(iqrs)
    # plot_timecourse(
    #     data.obs['sham_tmp'],
    #     compound_masks,
    #     plt.gca(),
    #     plot_shadow=False,suppress_legend=True,alpha=.3
    # )
# plt.show()


In [ ]:
# Let's establish empirical quantiles for injury vs ctrl delta everywhere + rgcs only
compound_masks = get_compound_masks(data)

time_deltas = []

for timecourse in sham_timecourses:
    deltas = [timecourse['injury'][t]['center'] - timecourse['control'][t]['center'] for t in [12,24,48,72]]
    time_deltas.append(deltas)

time_deltas = np.array(time_deltas)
time_deltas

In [ ]:
fig,axes = plt.subplots(1,4,figsize=(20,5))
for i,deltas,ax in zip(range(4),time_deltas.T,axes):
    ax.hist(deltas,bins=50)
    ax.set_title(f"Time {i+1}")
fig.tight_layout()
fig.show()


In [ ]:
delta_quantiles = {
    t:np.percentile(time_deltas[:,i],[np.arange(100)])[0] for i,t in enumerate([12,24,48,72])
}
delta_quantiles

In [ ]:
# plt.figure()

bound_ranges = [
    (25,75),
    (15,85),
    (5,95),
    (1,99)
]

plt.figure()
for lower,upper in bound_ranges:
    times = [12,24,48,72]
    upper_bound = np.array([delta_quantiles[t][upper] for t in times])
    lower_bound = np.array([delta_quantiles[t][lower] for t in times])
    plt.fill_between(
        times,upper_bound,lower_bound,
        label=f"{lower}-{upper}",alpha=.05,color='blue'
    )
plt.show()


# Let's plot a large number sans shadows actually. Just simple overlay
# compound_masks = get_compound_masks(data)

# sham_timecourses = []

# # fig,axes = plt.subplots(2,3,figsize=(10,6))
# for sham in small_sham_signature_masks:
#     sc.tl.score_genes(data,data.var_names[sham],score_name=f'sham_tmp')
#     sham_timecourses.append(data.obs['sham_tmp'])
    # target = data.obs['sham_tmp']
    # plot_timecourse(
    #     target,compound_masks=compound_masks,ax=axes[0,0],)
# for sham in medium_sham_signature_masks:
#     pass
# fig.tight_layout()
# fig.show()

In [ ]:
compound_masks = get_compound_masks(data)

fig,axes = plt.subplots(3,5,figsize=(20,10))
for i,ax in enumerate(axes.flatten()):
    target = data.obs[f'sham_{i}']
    plot_timecourse(
        target,compound_masks,ax,ax_label=f'Sham Signature {i}',
        central_tendency="mean"
    )
fig.tight_layout()
fig.show()

# Timecourse Plot

In [ ]:
# Let's plot a timecourse for each between injury and ctrl 
control_mask =  data.obs['exp_condition'] == "Cntr"
injury_mask = data.obs['exp_condition'] == "Mtz"

times = [12,24,48,72]

time_masks = [
    data.obs['time'] == t
    for t in times
]

# Timecourse with an IQR shadow

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    # We build the IQR shadows here as dictionaries

    time_control_iqrs = {}
    time_injury_iqrs = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_injury_mask = injury_mask & time_mask

        control_data = data.obs[score][combined_control_mask]
        injury_data = data.obs[score][combined_injury_mask]

        time_control_iqrs[t] = extract_iqrs(control_data)
        time_injury_iqrs[t] = extract_iqrs(injury_data)

    # Actual plotting
    ax = axes[i]
    filled_centerline_to_ax(
        ax,time_control_iqrs,
        label="control (median)",fill_label="control (IQR)",ax_label=f"{score}, (all cells)",
    )
    filled_centerline_to_ax(
        ax,time_injury_iqrs,
        label="injury (median)",fill_label="injury (IQR)",ax_label=f"{score}, (all cells)",
    )
    ax.legend(loc="upper left")

fig.tight_layout()
fig.show()

# Timecourse with an SEM 

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    # Build SEM shadows

    time_control_sem_range = {}
    time_injury_sem_range = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_injury_mask = injury_mask & time_mask

        control_target = data.obs[score][combined_control_mask]
        injury_target = data.obs[score][combined_injury_mask]

        time_control_sem_range[t] = extract_sems(control_target)
        time_injury_sem_range[t] = extract_sems(injury_target)

    # Actual plotting
    ax = axes[i]
    filled_centerline_to_ax(
        ax,time_control_sem_range,
        label="control (mean)",fill_label="control (SEM)",ax_label=f"{score}, (all cells)",
    )
    filled_centerline_to_ax(
        ax,time_injury_sem_range,
        label="injury (mean)",fill_label="injury (SEM)",ax_label=f"{score}, (all cells)",
    )
    ax.legend(loc="best")

fig.tight_layout()
fig.show()

### UMAP Score Plots

We can see that the distribution of the scores is similar but not fully overlapping. We can probably confirm this intuition with a bunch of pairwise scatters, but this basically tracks with the lit anyway afaik. 

In [ ]:
fig,axes = plt.subplots(1,3,figsize=(25,5))
axes = axes.flatten()

sc.pl.umap(data,color='parthanatos',ax=axes[0],show=False)
sc.pl.umap(data,color='necroptosis',ax=axes[1],show=False)
sc.pl.umap(data,color='apoptosis',ax=axes[2],show=False)

fig.show()

# Subset to Retinal Ganglion Cells

We want to check if the behavior of RGCs in particular is different than the rest of the pop with respect to the scores of interest.

We will isolate just the RGCs using leiden mapping, and then re-plot some of our previous analysis

In [ ]:
data.obs.keys()

In [ ]:
def plot_origin_axes(ax,plot_x=True,plot_y=True):
    xmin,xmax = ax.get_xlim()
    # xmin = min(0.,xmin)
    ymin,ymax = ax.get_ylim()
    # ymin = min(0.,ymin)
    if plot_x:
        ax.plot([xmin,xmax],[0,0],'k--')
    if plot_y:
        ax.plot([0,0],[ymin,ymax],'k--')
    return ax

ylims = {
    'parthanatos':(-.05,.1),
    'necroptosis':(-.05,.1),
    'apoptosis':(-.05,.1),
}

all_cell_types = sorted(data.obs['cell_type'].unique())

for cell_type in all_cell_types:

    if cell_type == "trash":
        continue

    cell_type_data = data[data.obs['cell_type'] == str(cell_type)]
    non_cell_type_data = data[data.obs['cell_type'] != str(cell_type)]

    compound_masks = get_compound_masks(cell_type_data)
    non_compound_masks = get_compound_masks(non_cell_type_data)
    # fig,axes = plt.subplots(2,3,figsize=(10,7))
    fig,axes = plt.subplots(1,3,figsize=(10,5))

    fig.suptitle(f"Cell Type {cell_type}")
    for j,score in enumerate(['parthanatos','necroptosis','apoptosis']):
        # ax = axes[0][j]
        ax = axes[j]
        target_scores = cell_type_data.obs[score]
        plot_timecourse(
            target_scores,compound_masks,ax,central_tendency="median",plot_shadow=False,
            label=cell_type
        )
        ax.set_title(f"{score} ({cell_type})")
        ax.set_ylim(*ylims[score])

        # ax = axes[1][j]
        target_scores = non_cell_type_data.obs[score]
        plot_timecourse(
            target_scores,non_compound_masks,ax,central_tendency="median",plot_shadow=False,
            label="Rest"
        )
        ax.set_title(f"{score}")
        ax.set_ylim(*ylims[score])

        plot_origin_axes(ax,plot_x=True,plot_y=False)

    fig.tight_layout()
    fig.show()


# RGC Vs Rest Comparison

Let's compare the RGCs to the rest of the population to observe any differences for our scores of interest

In [ ]:
rgc_mask = data.obs['cell_type'] == "RGCs"
rgc = data[rgc_mask]
non_rgc = data[~rgc_mask]

times = [12,24,48,72]

rgc_time_masks = [
    rgc.obs['time'] == t
    for t in times
]

non_rgc_time_masks = [
    non_rgc.obs['time'] == t
    for t in times
]

rgc_control_mask =  rgc.obs['exp_condition'] == "Cntr"
rgc_injury_mask = rgc.obs['exp_condition'] == "Mtz"

non_rgc_control_mask =  non_rgc.obs['exp_condition'] == "Cntr"
non_rgc_injury_mask = non_rgc.obs['exp_condition'] == "Mtz"

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_rgc_control_iqrs = {}
    time_rgc_injury_iqrs = {}
    time_non_rgc_control_iqrs = {}
    time_non_rgc_injury_iqrs = {}

    for t,(rgc_time_mask,non_rgc_time_mask) in zip(times,zip(rgc_time_masks,non_rgc_time_masks)):

        combined_rgc_control_mask = rgc_control_mask & rgc_time_mask
        combined_rgc_injury_mask = rgc_injury_mask & rgc_time_mask

        combined_non_rgc_control_mask = non_rgc_control_mask & non_rgc_time_mask
        combined_non_rgc_injury_mask = non_rgc_injury_mask & non_rgc_time_mask

        time_rgc_control_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_control_mask])
        time_rgc_injury_iqrs[t] = extract_iqrs(rgc.obs[score][combined_rgc_injury_mask])

        time_non_rgc_control_iqrs[t] = extract_iqrs(non_rgc.obs[score][combined_non_rgc_control_mask])
        time_non_rgc_injury_iqrs[t] = extract_iqrs(non_rgc.obs[score][combined_non_rgc_injury_mask])

    ax = axes[i]

    # We're going to discard all the IQR stuff here because the shadows make this more or less unreadable. 
    # But I do think it's important to visualize this in overlay to convey the idea

    ax.plot(times, [time_rgc_control_iqrs[t]['center'] for t in times], label="RGC Control", color='r')
    ax.plot(times, [time_rgc_injury_iqrs[t]['center'] for t in times], label="RGC injury", color='b')
    ax.plot(times, [time_non_rgc_control_iqrs[t]['center'] for t in times], label="Non-RGC Control", color='r', linestyle='--',alpha=.3)
    ax.plot(times, [time_non_rgc_injury_iqrs[t]['center'] for t in times], label="Non-RGC injury", color='b', linestyle='--',alpha=.3)

    ax.set_xlabel("Time (h)")
    ax.set_ylabel(f"{score}")
    ax.set_title(f"{score} (RGC vs Non-RGC)")
    ax.legend()

fig.tight_layout()
fig.show()


# PCA sanity check

We should check if any computed PCs meaningfully correspond to the computed cell death scores, as this would be a simple way to verify whether or not this is a relatively orthogonal signal

In [ ]:
corr = np.corrcoef(data.obsm['X_pca'].T,data.obs[['parthanatos','necroptosis','apoptosis']].T)[50:,:50]

plt.figure()
plt.imshow(
    corr.T,
    cmap='bwr',vmin=-1,vmax=1,
    aspect='auto'
)
plt.ylabel("PCs")
plt.xticks(np.arange(3),labels=['parthanatos','necroptosis','apoptosis'])
plt.colorbar(label="Pearson Correlation")
plt.title("Cell Death Scores vs PCA Scores")
plt.show()

In [ ]:
parthanatos_min,parthanatos_max = np.min(corr[0]),np.max(corr[0])
necroptosis_min,necroptosis_max = np.min(corr[1]),np.max(corr[1])
apoptosis_min,apoptosis_max = np.min(corr[2]),np.max(corr[2])

parthanatos_min,parthanatos_max = np.around(parthanatos_min,3),np.around(parthanatos_max,3)
necroptosis_min,necroptosis_max = np.around(necroptosis_min,3),np.around(necroptosis_max,3)
apoptosis_min,apoptosis_max = np.around(apoptosis_min,3),np.around(apoptosis_max,3)

print(f"Correlations: \t\tMin\t\tMax")
print("========================================================")
print(f"parthanatos \t\t{parthanatos_min},\t\t{parthanatos_max}")
print(f"necroptosis \t\t{necroptosis_min},\t\t{necroptosis_max}")
print(f"apoptosis \t\t{apoptosis_min},\t\t{apoptosis_max}")

Apoptosis seems to have popped out as the strongest but not by a massive amount. It has (negative) 33% correlation to PC 2

# Population Plot

In [ ]:
# Pairwise proportion plots
data.obs['leiden'].unique()

In [ ]:
from misc_utils import auto_split_range

def get_pairwise_proportions(data,n_leiden=20):

    # sizes = data.obs.groupby('leiden').size()
    sizes = np.array([np.sum(data.obs['leiden'] == str(i)) for i in range(n_leiden)])
    pairwise = np.outer((1+sizes),1/(1+sizes))
    return pairwise

pairwise = get_pairwise_proportions(data)

plt.figure()
plt.imshow(
    np.log(pairwise),
    **auto_split_range(np.log(pairwise))
)
plt.colorbar()
plt.show()

n_clusters = len(data.obs['leiden'].unique())

compound_ratios = {'injury':{},'control':{}}

compound_masks = get_compound_masks(data)
fig,axes = plt.subplots(2,4,figsize=(20,10))
for i,condition in enumerate(compound_masks):
    for j,time in enumerate(compound_masks[condition]):
        ax = axes[i][j]
        subset = data[compound_masks[condition][time]]
        pairwise = get_pairwise_proportions(subset,n_leiden=n_clusters)
        compound_ratios[condition][time] = pairwise
        im = ax.imshow(
            np.log(pairwise),
            **auto_split_range(np.log(pairwise),force_range=2)
        )
        plt.colorbar(im,ax=ax)
        ax.set_xticks(np.arange(n_clusters),labels=np.arange(n_clusters),rotation=90)
        ax.set_yticks(np.arange(n_clusters),labels=np.arange(n_clusters))
        ax.set_title(f"{condition} {time}")
fig.tight_layout()
fig.show()


In [ ]:
fig,axes = plt.subplots(1,4,figsize=(20,5))
fig.suptitle("Control / Injury Ratio of Ratios")
for time,ax in zip(compound_masks['control'],axes):
    ror = (compound_ratios['control'][time])/(compound_ratios['injury'][time])
    im = ax.imshow(
        np.log(ror),
        **auto_split_range(np.log(ror))
    )
    ax.set_xticks(np.arange(n_clusters),labels=np.arange(n_clusters),rotation=90)
    ax.set_yticks(np.arange(n_clusters))
    plt.colorbar(im,ax=ax)

fig.tight_layout()
fig.show()


In [ ]:
for a in compound_ratios['injury'].values():
    print(a.shape)

In [ ]:
# compound_masks = get_compound_masks(data)
# rgc_target_10 = data.obs['leiden'] == "10"
# rgc_target_3 = data.obs['leiden'] == "3"
# rgc_target_both = d

# compound_scores = get_compound_scores(compound_masks,rgc_target_10.astype(dtype=int))
# compound_means = get_compound_means(compound_scores)

target_7 = data.obs['leiden'] == "7"


plt.figure()
plot_timecourse(
    target_7.astype(int),compound_masks,plt.gca(),
    plot_shadow=False,central_tendency="mean", # ax_label=
)
plt.show()

In [ ]:
compound_masks = get_compound_masks(data)
clusters = sorted(list(set(data.obs['leiden'])),key=float)

fig,axes = plt.subplots(4,5,figsize=(15,15))
for cluster,ax in zip(clusters,axes.flatten()):
    cluster_target = data.obs['leiden'] == cluster
    plot_timecourse(
        cluster_target,compound_masks,ax,
        ax_label=cluster,plot_shadow=False
    )
fig.tight_layout()
fig.show()

In [ ]:
rgc_3.shape

In [ ]:
for time in [12,24,48,72]:
    print(f"{time}: {np.sum((rgc_3.obs['time'] == time) & (rgc_3.obs['exp_condition'] == "Cntr"))}")
    

In [ ]:
for time in [12,24,48,72]:
    print(f"{time}: {np.sum((rgc_3.obs['time'] == time) & (rgc_3.obs['exp_condition'] == "Mtz"))}")


In [ ]:
for time in [12,24,48,72]:
    print(f"Injury: {time}, {np.sum((data.obs['time'] == time) & (data.obs['exp_condition'] == "Mtz"))}")
for time in [12,24,48,72]:
    print(f"Control: {time}, {np.sum((data.obs['time'] == time) & (data.obs['exp_condition'] == "Cntr"))}")


In [ ]:
np.sum(data.obs['leiden'] == "19")

In [ ]:
sc.pl.dotplot(data[data.obs['exp_condition'] == "Cntr"],['parthanatos','necroptosis','apoptosis'],groupby="cell_type",figsize=(5,10))
sc.pl.dotplot(data[data.obs['exp_condition'] == "Mtz"],['parthanatos','necroptosis','apoptosis'],groupby="cell_type",figsize=(5,10))

In [ ]:
# pairwise